# 第 6 章 モジュール・エラー・アルゴリズム: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/economics-programming/06-modules-errors-and-algorithms.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
demand_intercept = 100  # 需要 Q = a - bP の切片 a
demand_slope = 2        # 需要の傾き b
supply_intercept = 0    # 供給 Q = c + dP の切片 c
supply_slope = 3        # 供給の傾き d
low = 0                 # 二分法で探す区間の下限（価格）
high = 100              # 二分法で探す区間の上限（価格）
tolerance = 1e-9        # 区間の幅が、これより小さくなったら、止める
steps = 6               # 図に描く、二分法の中点の数

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_lib = sys.modules.get("lib")
if _lib is not None and (
    Path(getattr(_lib, "__file__", "") or "").resolve().parent
    != _root / "lib"
):
    for _name in list(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 二分法で、均衡価格を求める

In [ ]:
from lib.chapter06.bisection import find_equilibrium_price, make_excess_demand

excess_demand = make_excess_demand(
    demand_intercept, demand_slope, supply_intercept, supply_slope
)
price = find_equilibrium_price(excess_demand, low, high, tolerance)
print(f"均衡価格: {price:.6f}")

## 図: 二分法の中点が、0 に近づく

In [ ]:
from lib.chapter06.plot import plot_bisection

plot_bisection(excess_demand, low, high, steps)

## 試してみよう

- `supply_intercept` を -10 にすると、均衡価格は 22 になります
- `low` を 30 にすると、区間 [30, 100] の両端で超過需要の符号が同じ（どちらも
  負）なので、`NoEquilibriumError` で止まります（均衡が区間の外にあるため）
- `tolerance` を大きくすると、反復が減ります（区間の幅を 100 から
  半分ずつに絞るとき、誤差 1e-9 では 37 回、1e-3 では 17 回、1 では 7 回です）